In [2]:
!pip install google-colab google-auth

import os
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
import pandas as pd
from google.colab import drive

# Funções de extração de dados
def extrair_dados_produto(produto):
    try:
        # Extrair o título do produto
        nome = produto.find('div', class_='item-title').get_text(strip=True)

        # Extrair o link do produto
        link = produto.find('div', class_='item-title').a['href']

        # Extrair o preço do produto
        preco = produto.find('span', class_='price').get_text(strip=True)

        # Extrair a descrição do produto
        descricao = produto.find('div', class_='desc').get_text(strip=True)

        # Extrair a URL da imagem do produto
        imagem_url = produto.find('div', class_='thumb-wrapper').find('img')['src']

        # Extrair o SKU do produto, se disponível
        sku = produto.find('div', class_='desc').find('p').get_text(strip=True) if produto.find('div', class_='desc').find('p') else '0'

        # Salvar os dados em um dicionário
        dados_produto = {
            'Fonte': 'FOGOES SHOP',
            'Data_Coleta': date.today().strftime('%Y-%m-%d'),
            'Titulo': nome,
            'Link': link,
            'Preco': preco,
            'Descricao': descricao,
            'Imagem_URL': imagem_url,
            'SKU': sku
        }

        return dados_produto
    except Exception as e:
        print(f"\033[91mErro ao extrair dados do produto: {e}\033[0m")
        return None

def salvar_dados_csv(dados_produtos, category):
    try:
        df = pd.DataFrame(dados_produtos)

        # Criar a pasta 'output' se ainda não existir
        output_folder = "output"
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
            print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

        # Salvar o arquivo CSV na pasta 'output'
        file_path = os.path.join(output_folder, f"{category}.csv")
        df.to_csv(file_path, index=False)
        print(f"\033[92mDados da categoria '{category}' salvado com sucesso\033[0m")
        print()

        # Adicionar informações específicas
        print(f"\033[92mARQUIVO CRIADO {date.today()}\033[0m")

    except Exception as e:
        print(f"\033[91mErro ao salvar dados em CSV: {e}\033[0m")

def extrair_dados(url, category):
    try:
        print(f"Processando categoria '{category}' - URL: {url}")
        dados_produtos = []

        # Iteramos sobre todas as páginas
        page_num = 1
        while True:
            page_url = f"{url}/page/{page_num}/" if page_num > 1 else url
            print(f"Navegando na página {page_num}...")
            response = requests.get(page_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            produtos = soup.find_all('li', class_='product-layout')

            if not produtos:
                break

            for produto in produtos:
                dados_produto = extrair_dados_produto(produto)
                if dados_produto:
                    dados_produtos.append(dados_produto)

            page_num += 1

        # Salvar os dados em CSV
        if dados_produtos:
            salvar_dados_csv(dados_produtos, category)

    except requests.RequestException as e:
        print(f"\033[91mErro ao fazer a requisição para a página {url}: {e}\033[0m")
    except Exception as e:
        print(f"\033[91mErro ao extrair dados da página {url}: {e}\033[0m")

# Função principal
def main():
    try:
        # Lista de links da Fogo E Shop com a categoria
        links = {
            "Acessorios Banheiro": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/",
"Modelos de Deca": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-deca/",
"Modelos Metalworks by Crismoe": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-crismoe-metalworks/",
"Sacaria": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/sacaria-areia-cimento-argamassa-outros/",
"Vedacao e impermeabilizacao": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/vedação-impermeabilizante/",
"Blocos e Tijolos": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/blocos-tijolos/",
"Papelao, Saco de Entulho e Lona": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/papelao-lona-saco-plastico/",
"Acos e Madeiras": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/aços-madeiras/",
"Equipamentos p/ Contrucao": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/equipamento-construcao/",
"Eletrodutos e Mangueiras": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/eletroduto-conduite-mangueira/",
"Ferramentas": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/marca-vonder/",
"Serra-Marmore": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/marca-vonder/Serra-Marmore/",
"Caixa d'Agua": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/caixa-agua-cisterna/",
"Tubo e Conexao": "https://fogoesshop.com.br/hidraulica-construcao/tubo-conexao/",
"Aquecedor gas": "https://fogoesshop.com.br/hidraulica-construcao/Aquecedor-gas/",
"Bombas e Pressurizadores": "https://fogoesshop.com.br/hidraulica-construcao/bombas-pressurizadores/",
"Ralo linear": "https://fogoesshop.com.br/hidraulica-construcao/ralo-linear/",
"Triturador Sanitario": "https://fogoesshop.com.br/hidraulica-construcao/triturador-sanitario-sanitrit/",
"Caixa de Descarga": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-caixa-descarga-embutida-deca/",
"Colecao Assinada": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-caixa-descarga-embutida-deca/",
"Cubas": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-cubas/",
"Louca p/ Banheiros": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-loucas-p-banheiros/",
"Metais p/ Banheiros": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-metais-p-banheiros/",
"Promocoes Banheiro, sale, Black friday": "https://fogoesshop.com.br/banheiro-construcao-reforma/promocoes-banheiro/",
"Revestir": "https://fogoesshop.com.br/banheiro-construcao-reforma/revestir/",
"Banheira": "https://fogoesshop.com.br/fshop-banho/banheira/",
"Duchas e Chuveiros": "https://fogoesshop.com.br/fshop-banho/fshop-duchas/",
"Chuveiro Eletrico": "https://fogoesshop.com.br/fshop-banho/chuveiro-eletrico/",
"Mekal": "https://fogoesshop.com.br/fshop-cozinha/cuba-inox-mekal/",
"Tramontina": "https://fogoesshop.com.br/fshop-cozinha/fshop-tramontina/",
"Torneira Eletrica": "https://fogoesshop.com.br/fshop-cozinha/torneira-eletrica/",
"Metais p/ Cozinha": "https://fogoesshop.com.br/fshop-cozinha/fshop-metais-p-cozinhas/",
"Promocoes Cozinha, sale, Black Friday": "https://fogoesshop.com.br/fshop-cozinha/promocoes-cozinha/",
"Torneira filtro": "https://fogoesshop.com.br/fshop-cozinha/fshop-torneira-filtro-e-filtro/",
"Iluminacao": "https://fogoesshop.com.br/fshop-eletrica/iluminacao-lampada-luminaria/",
"Tomadas e Interruptores": "https://fogoesshop.com.br/fshop-eletrica/tomada-interruptor-placa/",
"Fios e Cabos": "https://fogoesshop.com.br/fshop-eletrica/cabos-flexivel-sil/",
"Conduletes e Conexoes": "https://fogoesshop.com.br/fshop-eletrica/conduletes/",
"Disjuntores e Armacoes": "https://fogoesshop.com.br/fshop-eletrica/disjuntores/",
"Promocoes Tintas, sales tinta": "https://fogoesshop.com.br/fshop-tintas/promocoes-tintas/",
"Pre-Pintura": "https://fogoesshop.com.br/fshop-tintas/pre-pintura/",
"Ferramentas E Acessorios": "https://fogoesshop.com.br/fshop-tintas/ferramentas-acessorios-trinchas-rolos/",
"Adesivos, Colas, Selantes e Silicones": "https://fogoesshop.com.br/fshop-tintas/Adesivos-colas-selantes-silicones/",
"Solventes e Removedores": "https://fogoesshop.com.br/fshop-tintas/solventes-removedores/",
"Acessibilidade": "https://fogoesshop.com.br/fshop-outros/fshop-ACESSIBILIDADE/",
"Complementares": "https://fogoesshop.com.br/fshop-outros/fshop-complementares/",
"Jardim": "https://fogoesshop.com.br/fshop-outros/jardim-amanco/",
"Tanques e Area Externa": "https://fogoesshop.com.br/fshop-outros/fshop-tanques-area-externa/",
"Outlet": "https://fogoesshop.com.br/fshop-outros/outlet/"
        }

        # Iterar sobre as categorias e extrair dados
        for category, url in links.items():
            extrair_dados(url, category)

    except Exception as e:
        print(f"\033[91mErro no processo principal: {e}\033[0m")

if __name__ == "__main__":
    main()


Processando categoria 'Acessorios Banheiro' - URL: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/
Navegando na página 1...
Navegando na página 2...
Navegando na página 3...
Navegando na página 4...
Navegando na página 5...
Navegando na página 6...
Navegando na página 7...
Navegando na página 8...
Navegando na página 9...
Navegando na página 10...
Navegando na página 11...
Navegando na página 12...
Navegando na página 13...
Navegando na página 14...
Navegando na página 15...
Navegando na página 16...
Navegando na página 17...
Navegando na página 18...
A pasta output foi criada.
Dados da categoria 'Acessorios Banheiro' salvado com sucesso

ARQUIVO CRIADO 2024-05-17
Processando categoria 'Modelos de Deca' - URL: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-deca/
Navegando na página 1...
Navegando na página 2...
Navegando na página 3...
Navegando na página 4...
Navegand

In [3]:
!pip install google-colab google-auth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.2 MB/s eta 0:00:00


In [7]:
!pip install --upgrade google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.2/189.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 26.9 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.27.0
    Uninstalling google-auth-2.27.0:
      Successfully uninstalled google-auth-2.27.0
  Attempting uninstall: google-auth-httplib2
    Found existing installation: google-auth-httplib2 0.1.1
    Uninstalling google-auth-httplib2-0.1.1:
      Successfully uninstalled google-auth-httplib2-0.1.1
  Attempting uninstall: google-api-python-client
    Found existing installation: google-api-python-client 2.84.0
    Uninstalling google-api-python-client-2.84.0:
      Successfully uninstalled google-api-python-client-2.84.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.27.0, but

In [6]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import date
import pandas as pd
from google.colab import drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle

# Montar o Google Drive
drive.mount('/content/drive')

# Funções de extração de dados
def extrair_dados_produto(produto):
    try:
        nome = produto.find('div', class_='item-title').get_text(strip=True)
        link = produto.find('div', class_='item-title').a['href']
        preco = produto.find('span', class_='price').get_text(strip=True)
        descricao = produto.find('div', class_='desc').get_text(strip=True)
        imagem_url = produto.find('div', class_='thumb-wrapper').find('img')['src']
        sku = produto.find('div', class_='desc').find('p').get_text(strip=True) if produto.find('div', 'desc').find('p') else '0'

        dados_produto = {
            'Fonte': 'FOGOES SHOP',
            'Data_Coleta': date.today().strftime('%Y-%m-%d'),
            'Titulo': nome,
            'Link': link,
            'Preco': preco,
            'Descricao': descricao,
            'Imagem_URL': imagem_url,
            'SKU': sku
        }

        return dados_produto
    except Exception as e:
        print(f"Erro ao extrair dados do produto: {e}")
        return None

def salvar_dados_csv(dados_produtos, category):
    try:
        df = pd.DataFrame(dados_produtos)

        output_folder = "output"
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
            print(f"A pasta {output_folder} foi criada.")

        file_path = os.path.join(output_folder, f"{category}.csv")
        df.to_csv(file_path, index=False)
        print(f"Dados da categoria '{category}' salvos com sucesso")
        print(f"ARQUIVO CRIADO {date.today()}")

        enviar_para_drive(file_path, f"{category}.csv")

    except Exception as e:
        print(f"Erro ao salvar dados em CSV: {e}")

def enviar_para_drive(local_file_path, remote_file_name):
    try:
        SCOPES = ['https://www.googleapis.com/auth/drive.file']
        creds = None

        if os.path.exists('token.pickle'):
            with open('token.pickle', 'rb') as token:
                creds = pickle.load(token)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
                creds = flow.run_local_server(port=0)

            with open('token.pickle', 'wb') as token:
                pickle.dump(creds, token)

        service = build('drive', 'v3', credentials=creds)

        file_metadata = {'name': remote_file_name}
        media = MediaFileUpload(local_file_path, mimetype='text/csv')
        file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        print(f"Arquivo {remote_file_name} enviado para o Google Drive com sucesso, ID: {file.get('id')}")

    except Exception as e:
        print(f"Erro ao enviar o arquivo para o Google Drive: {e}")

def extrair_dados(url, category):
    try:
        print(f"Processando categoria '{category}' - URL: {url}")
        dados_produtos = []

        page_num = 1
        while True:
            page_url = f"{url}/page/{page_num}/" if page_num > 1 else url
            print(f"Navegando na página {page_num}...")
            response = requests.get(page_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            produtos = soup.find_all('li', class_='product-layout')

            if not produtos:
                break

            for produto in produtos:
                dados_produto = extrair_dados_produto(produto)
                if dados_produto:
                    dados_produtos.append(dados_produto)

            page_num += 1

        if dados_produtos:
            salvar_dados_csv(dados_produtos, category)

    except requests.RequestException as e:
        print(f"Erro ao fazer a requisição para a página {url}: {e}")
    except Exception as e:
        print(f"Erro ao extrair dados da página {url}: {e}")

def main():
    try:
        links = {
            "Acessorios Banheiro": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/",
            # Adicione outras categorias e URLs aqui
        }

        for category, url in links.items():
            extrair_dados(url, category)

    except Exception as e:
        print(f"Erro no processo principal: {e}")

if __name__ == "__main__":
    main()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 47)

In [ ]:
from google.colab import files
files.upload()  # Faça o upload do seu arquivo credentials.json
